# Reconcile source against target

Compares what the target holds against what discovery recorded and reports the verdict per table. Read-only on both ends.

---

*Generated from `engine/dataplane/03_reconcile.py` by `engine/target/stage_notebooks.py`. Regenerate with `snowmig.py build-notebooks`; do not hand-edit — an edit here is overwritten on the next build. Change the source instead.*

In [ ]:
# ── PARAMETERS ─────────────────────────────────────────────────
# Edit these, then run the notebook top to bottom. Every value is a
# plain Python literal: `None` means the flag is not passed at all,
# and `True` means a bare switch is passed.
#
# Scope and mode are INPUTS. To migrate less, change a value here --
# never edit the stage logic below to make it cover less.
PARAMS = {
    'target-catalog': None,  # REQUIRED
    'ddl-plan': None,
    'reports-dir': None,
    'output-dir': None,
    'counts': False,
}

# ── WORKFLOW PARAMETERS WIN ────────────────────────────────────
# A job task's `parameters` ({name, value}) override the literals
# above, by the same names (`schema`, `mode`, `dry-run`, ...). Read
# with oidlUtils.parameters.getParameter -- resolved by the AIDP
# runtime, never imported. The environment is only a fallback. This
# is how ONE notebook serves one workflow per schema: each task
# passes its own `schema`.
import os as _os
_MISS = '\x00__no_parameter__'
_SWITCHES = ['counts']
_MULTI = []
_CHOICES = {}
_TRUE = ('true', 'yes', 'on', '1')
_FALSE = ('false', 'no', 'off', '0', '')


def _spellings(name):
    # `dry-run` as an operator may type it in the console: dry-run,
    # dry_run, dryRun, dryrun, and the upper-case forms. A spelling
    # nobody reads is a silent default -- for dry-run, a real write.
    parts = name.split('-')
    snake = '_'.join(parts)
    camel = parts[0] + ''.join(p[:1].upper() + p[1:] for p in parts[1:])
    flat = ''.join(parts)
    return list(dict.fromkeys((name, snake, camel, flat, name.upper(),
                               snake.upper(), flat.upper())))


def _workflow_param(name):
    for n in _spellings(name):
        try:
            v = oidlUtils.parameters.getParameter(n, _MISS)  # noqa: F821
        except NameError:  # outside AIDP: no oidlUtils
            v = _MISS
        if v != _MISS and v is not None and str(v).strip() != '':
            return str(v).strip()
    # The environment is a fallback only, by the plain spellings: a
    # generic name (SCHEMA, DRYRUN) could be any stray cluster variable.
    for n in dict.fromkeys((name, name.replace('-', '_'),
                            name.replace('-', '_').upper())):
        v = _os.environ.get(n)
        if v is not None and str(v).strip() != '':
            return str(v).strip()
    return None


PARAMS_FROM_WORKFLOW = {}
for _key in list(PARAMS):
    _v = _workflow_param(_key)
    if _v is None:
        continue
    if _key in _SWITCHES:
        if _v.lower() in _TRUE:
            _v = True
        elif _v.lower() in _FALSE:
            _v = False
        else:
            # A typo is not guessed either way: read as False, a
            # `dry-run` typo would turn a dry run into a real write.
            raise ValueError(
                f'workflow parameter {_key}={_v!r}: `{_key}` is a '
                f'switch; give true or false')
    elif _key in _MULTI:
        _v = [p.strip() for p in _v.split(',') if p.strip()]
    _bad = [c for c in (_v if isinstance(_v, list) else [_v])
            if _key in _CHOICES and c not in _CHOICES[_key]]
    if _bad:
        raise ValueError(
            f'workflow parameter {_key}={_bad[0]!r}: `{_key}` takes '
            f"one of {', '.join(_CHOICES[_key])}")
    PARAMS[_key] = PARAMS_FROM_WORKFLOW[_key] = _v
print('from the workflow:', PARAMS_FROM_WORKFLOW)


def _argv(params, repeated=()):
    """PARAMS -> argv. None is omitted; True is a bare switch; a list
    follows its flag, or repeats the flag per value for a name in
    `repeated` (an append-style flag)."""
    argv = []
    for key, value in params.items():
        if value is None or value is False:
            continue
        if value is True:
            argv.append(f'--{key}')
            continue
        values = value if isinstance(value, (list, tuple)) else [value]
        if key in repeated:
            for v in values:
                argv.extend((f'--{key}', str(v)))
        else:
            argv.append(f'--{key}')
            argv.extend(str(v) for v in values)
    return argv


ARGV = _argv(PARAMS)
print('arguments:', ARGV)

_missing = [k for k in ['target-catalog'] if not PARAMS.get(k)]
if _missing:
    raise ValueError(
        f'set these PARAMS before running: {_missing}')

## Shared source helpers

Inlined from `engine/dataplane/snowmig_source.py` so this notebook runs with nothing else uploaded beside it.

In [ ]:
"""How the migration scripts READ Snowflake from inside AIDP. Two modes.

LIVE-VERIFIED 2026-09-16 on a real cluster (Spark 3.5, AIDP 4.x):

  connector (default)  spark.read.format("aidataplatform") with
                       type=SNOWFLAKE. Talked to the account, read a table
                       (87 rows x 9 cols) and ran a pushdown query
                       (`current_user()` = the service user). Needs NO extra
                       cluster library — the format is built in — and needs
                       NO successful catalog crawl.

  external-catalog     three-part names `catalog.schema.table` against a
                       registered EXTERNAL catalog. Cheaper (no per-read
                       Snowflake login) but it can only see what the CRAWLER
                       has already discovered, and on the validated
                       deployment the crawler failed with
                       "CONNECTOR_0067 ... Login has timed out" while the
                       connector above worked with the same credentials.

So: connector mode is the default because it is the one proven end to end,
and external-catalog mode is kept for a deployment whose crawl succeeds.

The option NAMES are the live-verified raw ones, which differ from the
Python helper's keyword names — `user.name` (not `user`), `database.name`,
`authentication.method` ∈ {Basic, KeyPair}, `private.key.content`. A wrong
name fails loud (`DATA_ACCESS_LAYER_0001 - Required spark option ... was not
provided`), which is how these were established.

Credentials come from a CONFIG FILE, never from arguments: the same rule the
control plane follows. Point --source-config at a JSON file shaped like
snowmig-config.example.yaml (JSON, on the workspace mount).
"""
from __future__ import annotations

import gzip
import hashlib
import json
import pathlib
import re

__all__ = ["SOURCE_MODES", "SourceConfigError", "SnowflakeSource",
           "write_step_output", "read_plan_json",
           "load_source_config"]

SOURCE_MODES = ("connector", "external-catalog")

AIDP_FORMAT = "aidataplatform"


# The verbs this transport may send. Deliberately narrower than the
# control-plane transport's list: WITH is absent, because following a CTE to
# the statement it prefixes needs the engine's lexer, which does not exist on
# a cluster. A read that needs a CTE can be written as a subquery.
PUSHDOWN_READ_VERBS = ("SELECT", "SHOW", "DESCRIBE", "DESC", "EXPLAIN")


class SourceWriteRefused(PermissionError):
    """A statement that is not a read was handed to the pushdown transport."""


def _code_only(sql: str) -> str:
    """`sql` with string literals and comments blanked, length preserved.

    A `;` inside a literal is data, not a statement boundary, and `--` or
    `//` inside one is not a comment. Blanking rather than deleting keeps offsets, so the
    scan cannot be confused about where anything starts.

    Lexed as SNOWFLAKE lexes: a backslash escapes only inside a '...'
    string; a "..." identifier ends at the first quote that is not doubled.
    Honouring a backslash there too made `"a\\"; delete from T` one
    identifier to this scan and two statements to Snowflake -- the guard
    failed open.
    """
    out = []
    i, n = 0, len(sql)
    while i < n:
        c = sql[i]
        two = sql[i:i + 2]
        if c in ("'", '"'):
            out.append(" ")
            i += 1
            closed = False
            while i < n:
                if c == "'" and sql[i] == "\\" and i + 1 < n:  # escape
                    out.append("  ")
                    i += 2
                    continue
                if sql[i] == c:
                    if sql[i:i + 2] == c * 2:           # doubled = literal
                        out.append("  ")
                        i += 2
                        continue
                    out.append(" ")
                    i += 1
                    closed = True
                    break
                out.append("\n" if sql[i] == "\n" else " ")
                i += 1
            if not closed:
                raise SourceWriteRefused(
                    f"an unclosed {c} literal; refused (fails closed)")
            continue
        if two == "$$":
            out.append("  ")
            i += 2
            while i < n and sql[i:i + 2] != "$$":
                out.append("\n" if sql[i] == "\n" else " ")
                i += 1
            if i >= n:
                raise SourceWriteRefused(
                    "an unclosed $$ string; refused (fails closed)")
            out.append("  ")
            i += 2
            continue
        # `//` is a line comment in Snowflake exactly as `--` is. Missing it
        # let an apostrophe in `// it's` open a phantom literal that hid the
        # `;` on the next line -- two statements read as one.
        if two in ("--", "//"):
            while i < n and sql[i] != "\n":
                out.append(" ")
                i += 1
            continue
        if two == "/*":
            # Snowflake ends a block comment at the FIRST `*/` (no nesting),
            # as the engine's lexer does. Counting depth made
            # `select 1 /* /* */ ; delete from t` one statement here and
            # two to Snowflake. An unclosed comment is refused.
            i += 2
            out.append("  ")
            while i < n and sql[i:i + 2] != "*/":
                out.append("\n" if sql[i] == "\n" else " ")
                i += 1
            if i >= n:
                raise SourceWriteRefused(
                    "an unclosed /* comment; refused (fails closed)")
            out.append("  ")
            i += 2
            continue
        out.append(c)
        i += 1
    return "".join(out)


def assert_pushdown_read_only(sql: str) -> None:
    """Refuse anything that is not a single read. Fails closed.

    The cluster-side counterpart of the control plane's `assert_read_only`:
    the credential the notebook holds may well be able to write, and the
    only thing standing between a migration and a modified SOURCE is this
    check.
    """
    code = _code_only(sql or "")
    if "->>" in code:
        # Snowflake's flow operator chains another statement into the same
        # request: `select 1 ->> delete from t` would pass a leading-verb
        # check. Nothing this plugin generates uses it.
        raise SourceWriteRefused(
            "the ->> flow operator chains statements; refused")
    statements = [s for s in code.split(";") if s.strip()]
    if not statements:
        raise SourceWriteRefused(
            f"empty statement refused; this transport is read-only against "
            f"Snowflake (allowed: {', '.join(PUSHDOWN_READ_VERBS)})")
    if len(statements) > 1:
        raise SourceWriteRefused(
            f"{len(statements)} statements in one pushdown; refused. A "
            f"second statement is how a write rides along behind a read.")
    verb = statements[0].split()[0].upper() if statements[0].split() else ""
    if verb == "WITH":
        raise SourceWriteRefused(
            "a CTE is refused by this transport: deciding whether `WITH ... "
            "INSERT` is a read needs the engine's scanner, which does not "
            "run on the cluster. Write the CTE as a subquery.")
    if verb not in PUSHDOWN_READ_VERBS:
        raise SourceWriteRefused(
            f"{verb or 'unrecognised statement'} refused: this transport is "
            f"read-only against Snowflake, whatever the credential allows "
            f"(allowed: {', '.join(PUSHDOWN_READ_VERBS)}).")


class SourceConfigError(ValueError):
    """The source config is missing, unreadable or incomplete."""


def q(identifier: str) -> str:
    """Backtick-quote one Spark identifier."""
    return "`" + str(identifier).replace("`", "``") + "`"


def _sql_ident(identifier: str) -> str:
    """Double-quote one SNOWFLAKE identifier (the pushdown runs there)."""
    return '"' + str(identifier).replace('"', '""') + '"'


# An identifier Snowflake reads WITHOUT quotes, and so resolves upper-cased.
_UNQUOTED_IDENT = re.compile(r"^[A-Za-z_][A-Za-z0-9_$]*$")


def _database_name(name: str) -> str:
    """The name Snowflake resolves the config's database to, unquoted.

    `database: snowmig_db` is an unquoted identifier to the connector, so it
    names SNOWMIG_DB. `database: '"MyDb"'` -- written with its double quotes,
    as Snowflake SQL would -- names the mixed-case MyDb exactly. Any other
    value that is not a plain identifier can only have been meant verbatim.
    Every statement that names the database goes through here, so no two
    of them can address different databases.
    """
    text = str(name).strip()
    if len(text) >= 2 and text[0] == text[-1] == '"':
        return text[1:-1].replace('""', '"')
    if _UNQUOTED_IDENT.match(text):
        return text.upper()
    return text


def _sql_database(name: str) -> str:
    """The config's database, quoted as the name Snowflake resolves it to."""
    return _sql_ident(_database_name(name))


# The copy expressions for a column whose live type is not the planned one,
# used only under `mapping.source_type_drift: convert`. A mirror of
# snowflake_source/dialect/types.copy_expressions -- this module is inlined
# into every notebook and cannot import the engine -- held to it by a parity
# test. The TIME precision is the live read's default (all nine digits);
# GEOGRAPHY reads as GeoJSON, as a plan without a recorded mode does.
_DRIFT_NUMERIC = {"NUMBER", "DECIMAL", "NUMERIC", "INT", "INTEGER", "BIGINT",
                  "SMALLINT", "TINYINT", "BYTEINT"}
_DRIFT_FLOATS = {"FLOAT", "FLOAT4", "FLOAT8", "DOUBLE", "REAL",
                 "DOUBLE PRECISION"}
_DRIFT_ZONED = ("TIMESTAMP_TZ", "TIMESTAMP_LTZ", "TIMESTAMP")
_DRIFT_JSON = ("VARIANT", "OBJECT", "ARRAY", "MAP")
_DRIFT_GEO = ("GEOGRAPHY", "GEOMETRY")


def live_copy_expressions(data_type: str, target_type: str, *,
                          name: str) -> tuple[str, str]:
    """(read_expr, convert_expr) for `name` read as its LIVE `data_type`
    and converted to the EXISTING target column's `target_type`."""
    col = '"' + str(name).replace('"', '""') + '"'
    out = "`" + str(name).replace("`", "``") + "`"
    key = str(data_type or "").strip().upper().split("(")[0].strip()
    target = str(target_type or "")
    upper = target.upper()
    if upper.startswith(("ARRAY<", "MAP<", "STRUCT<")):
        read = f"{col}::ARRAY::VARCHAR" if key == "VECTOR" \
            else f"{col}::VARIANT::VARCHAR"
        schema = target if "STRUCT<" in upper else target.lower()
        literal = "'" + schema.replace("\\", "\\\\").replace("'", "\\'") + "'"
        return read, f"from_json({out}, {literal})"
    if key in _DRIFT_NUMERIC:
        return f"{col}::VARCHAR", f"CAST({out} AS {target})"
    if key in _DRIFT_FLOATS:
        return f"TO_VARCHAR({col}, 'TME')", f"CAST({out} AS {target})"
    if key == "TIME":
        return f"TO_VARCHAR({col}, 'HH24:MI:SS.FF9')", out
    if key == "TIMESTAMP_NTZ":
        return (f"TO_VARCHAR({col}, 'YYYY-MM-DD HH24:MI:SS.FF9')",
                f"CAST({out} AS {target})")
    if key in _DRIFT_ZONED:
        return (f"TO_VARCHAR({col}, 'YYYY-MM-DD\"T\"HH24:MI:SS.FF9TZH:TZM')",
                f"CAST({out} AS {target})")
    if key in _DRIFT_GEO:
        return f"ST_ASGEOJSON({col})::VARCHAR", out
    if key in _DRIFT_JSON and upper == "STRING":
        return f"TO_JSON({col}::VARIANT)", out
    return col, out


def _sql_literal(value: str) -> str:
    """Escape a string literal for Snowflake SQL.

    The backslash first: Snowflake reads it as an escape inside '...', so a
    table named `a\\` made `'a\\'` an unterminated literal that swallowed the
    SQL after it.
    """
    return str(value).replace("\\", "\\\\").replace("'", "''")


# The /Workspace mount can serve a report that was just written before its
# bytes are all there: an incomplete read is retried briefly, then fails
# loudly, naming the file. Shared by every stage that re-reads a report.
REPORT_READ_TRIES = 5
REPORT_READ_WAIT = 2.0


def read_report_json(path: pathlib.Path) -> dict:
    """A report's JSON, retried briefly when the read comes back incomplete;
    still invalid after that is a loud failure that names the file."""
    import time
    last = None
    for attempt in range(REPORT_READ_TRIES):
        try:
            return json.loads(pathlib.Path(path).read_text(encoding="utf-8"))
        except json.JSONDecodeError as exc:
            last = exc
            if attempt + 1 < REPORT_READ_TRIES:
                time.sleep(REPORT_READ_WAIT)
    raise ValueError(f"{pathlib.Path(path).name} is not valid JSON after "
                     f"{REPORT_READ_TRIES} reads ({last}); it was not "
                     f"overwritten -- inspect it before re-running")


def load_source_config(path: str | pathlib.Path) -> dict:
    """Read the Snowflake connection config (JSON) from the workspace."""
    p = pathlib.Path(path).expanduser()
    try:
        text = p.read_text(encoding="utf-8")
    except OSError as exc:
        raise SourceConfigError(
            f"source config not readable at {p}: {exc.strerror}") from exc
    if p.suffix.lower() in (".yaml", ".yml"):
        try:
            import yaml
        except ImportError as exc:
            raise SourceConfigError(
                f"{p} is YAML but PyYAML is not on the cluster; write the "
                f"config as JSON instead") from exc
        try:
            data = yaml.safe_load(text) or {}
        except yaml.YAMLError as exc:
            # The parser's message quotes the offending line, which can be
            # the password line: only its position is reported.
            mark = getattr(exc, "problem_mark", None)
            where = f" at line {mark.line + 1}" if mark is not None else ""
            raise SourceConfigError(
                f"{p} is not valid YAML{where}; the line is withheld, "
                f"since it may hold the credential") from None
    else:
        try:
            data = json.loads(text) if text.strip() else {}
        except json.JSONDecodeError as exc:
            raise SourceConfigError(
                f"{p} is not valid JSON at line {exc.lineno}; the line is "
                f"withheld, since it may hold the credential") from None
    if not isinstance(data, dict):
        raise SourceConfigError(f"{p}: expected a mapping at the top level")
    # THE MIGRATION CONFIG IS ONE FILE FOR BOTH ENDS: the Snowflake connection
    # nested under `snowflake:`, the AIDP coordinates under `aidp:`. That is
    # the file `provision --source-config` reads, and it uploads the
    # `snowflake:` block as JSON (`plan/<stem>.json`) -- so the shape that
    # reaches the mount is the nested one, and reading the top level for
    # `account` found nothing but the envelope key. The failure surfaced on
    # the cluster, as every required field missing at once, which reads
    # like a broken credential rather than a config one level too deep.
    nested = data.get("snowflake")
    if isinstance(nested, dict):
        return dict(nested)
    return data


class SnowflakeSource:
    """Read-only access to the Snowflake source, in either mode.

    Nothing here can write: every method issues a read, and the connector is
    read-only in AIDP 4.0 by Oracle's own statement.
    """

    def __init__(self, spark, *, mode: str = "connector",
                 config: dict | None = None,
                 external_catalog: str | None = None,
                 session_schema: str | None = None):
        if mode not in SOURCE_MODES:
            raise SourceConfigError(
                f"unknown source mode {mode!r}; expected one of "
                f"{list(SOURCE_MODES)}")
        self.spark = spark
        self.mode = mode
        self.external_catalog = external_catalog
        self._options: dict[str, str] = {}
        # A REAL schema, used only to scope a pushdown session. The connector
        # validates this option against the schemas it can see and rejects
        # INFORMATION_SCHEMA itself with DATA_ACCESS_LAYER_0031 -- but a
        # pushdown query scoped to a real schema may reference
        # INFORMATION_SCHEMA freely (both established live).
        self.session_schema = session_schema or (config or {}).get("schema")

        if mode == "external-catalog":
            if not external_catalog:
                raise SourceConfigError(
                    "external-catalog mode needs --source-catalog")
            return

        cfg = config or {}
        missing = [k for k in ("account", "warehouse", "database", "user",
                               "auth") if not cfg.get(k)]
        if missing:
            raise SourceConfigError(
                "source config is missing required field(s): "
                + ", ".join(sorted(missing)))
        account = str(cfg["account"]).strip()
        # The connector wants the account/server URL host, which is what the
        # Snowflake console calls "Account/Server URL".
        host = str(cfg.get("host")
                   or f"{account}.snowflakecomputing.com").strip()
        opts = {
            "type": "SNOWFLAKE",
            "host": host,
            "port": str(cfg.get("port") or 443),
            "database.name": str(cfg["database"]).strip(),
            "user.name": str(cfg["user"]).strip(),
            "warehouse": str(cfg["warehouse"]).strip(),
        }
        if cfg.get("role"):
            opts["role"] = str(cfg["role"]).strip()

        # A secret may be inline in the one config file, or in a file the
        # config points at. On a cluster the inline form is usually the only
        # one available, since the workspace mount carries the config but not
        # the operator's home directory.
        def secret(inline, path_field):
            if cfg.get(inline):
                return str(cfg[inline])
            if cfg.get(path_field):
                return pathlib.Path(
                    str(cfg[path_field])).expanduser().read_text(encoding="utf-8").strip()
            return None

        auth = str(cfg["auth"]).strip().lower()
        if auth == "keypair":
            key = secret("private_key", "key_path")
            if not key:
                raise SourceConfigError(
                    "auth: keypair needs `private_key` (inline) or `key_path`")
            opts["authentication.method"] = "KeyPair"
            opts["private.key.content"] = key
            passphrase = secret("key_passphrase", "key_passphrase_path")
            if passphrase:
                opts["private.key.pass.phrase"] = passphrase
        elif auth == "password":
            password = secret("password", "password_path")
            if not password:
                raise SourceConfigError(
                    "auth: password needs `password` (inline) or "
                    "`password_path`")
            opts["authentication.method"] = "Basic"
            opts["password"] = password
        else:
            raise SourceConfigError(
                f"auth {auth!r} is not supported by the AIDP Snowflake "
                f"connector; use keypair (preferred) or password")
        self._options = opts

    # -- describing the estate ------------------------------------------

    def database(self) -> str | None:
        return self._options.get("database.name")

    def qualified(self, schema: str, table: str) -> str:
        """`"DB"."SCHEMA"."TABLE"` for a pushdown, the database from the config.

        LIVE 2026-09-29: the pushdown session has NO current schema, whatever
        the connector's `schema` option says, so an unqualified `"T"` fails
        with "Object does not exist" -- the root cause of the batched count's
        CONNECTOR_0099. Every table a pushdown names goes through here.
        Schema and table keep their exact case: they come from
        INFORMATION_SCHEMA, where the case is the name.
        """
        database = self.database()
        if not database:
            raise SourceConfigError(
                "a qualified pushdown needs the source database; add "
                "`database:` to the source config")
        return (f"{_sql_database(database)}.{_sql_ident(schema)}."
                f"{_sql_ident(table)}")

    def pushdown(self, sql: str, *, schema: str | None = None):
        """Run `sql` IN SNOWFLAKE and return a DataFrame.

        Refuses anything that is not a single read statement, whatever the
        credential allows -- see `assert_pushdown_read_only`.

        Connector mode only. The `schema` option must name a REAL schema:
        the connector rejects INFORMATION_SCHEMA there with
        DATA_ACCESS_LAYER_0031, while a query scoped to a real schema may
        reference INFORMATION_SCHEMA freely. Both established live.
        """
        # Before the mode check and before anything reaches Spark: the
        # refusal must not depend on configuration being right.
        assert_pushdown_read_only(sql)
        if self.mode != "connector":
            raise SourceConfigError(
                "pushdown is connector-mode only; external-catalog mode has "
                "no Snowflake session to push into")
        scope = schema or self.session_schema
        if not scope:
            raise SourceConfigError(
                "connector pushdown needs a REAL schema to scope the "
                "session: add `schema:` to the source config or pass "
                "--session-schema. INFORMATION_SCHEMA is not accepted there.")
        return (self.spark.read.format(AIDP_FORMAT)
                .options(**self._options)
                .option("schema", scope)
                .option("pushdown.sql", sql)
                .load())

    def source_counts(self, schema: str, tables: list[str], *,
                      chunk: int = 50) -> dict[str, int]:
        """`{table: COUNT(*)}` for many tables in ONE round trip per chunk.

        Connector mode opens a Snowflake session per read, and the copy needs
        a source count twice per table (before, and again after, to catch a
        source that moved during the copy). Per-table counting therefore cost
        more session setup than the copy itself on a live run. A single
        UNION ALL answers a whole schema instead; it is chunked because a
        statement with thousands of branches is its own problem.

        Every branch names its table "DB"."SCHEMA"."TABLE" (see `qualified`):
        unqualified, the live pushdown session refused every chunk with
        CONNECTOR_0099 and the copy fell back to a count per table.
        Qualified, 50 tables answered in one query in 25 s (2026-09-29).

        External-catalog mode has no session to amortise, so it falls back to
        a count per table -- correct either way, and the caller does not care.
        """
        out: dict[str, int] = {}
        if self.mode != "connector":
            for table in tables:
                out[table] = self.read_table(schema, table).count()
            return out
        for start in range(0, len(tables), chunk):
            batch = tables[start:start + chunk]
            sql = " union all ".join(
                # The literal is the table's own name, so one query can carry
                # many counts and still say which is which. Both the literal
                # and the identifier are escaped: one apostrophe in a table
                # name would otherwise break the whole chunk.
                f"select '{_sql_literal(t)}' as SNOWMIG_TABLE, "
                f"count(*) as SNOWMIG_N from {self.qualified(schema, t)}"
                for t in batch)
            for row in self.pushdown(sql, schema=schema).collect():
                data = row.asDict()
                out[str(data["SNOWMIG_TABLE"])] = int(data["SNOWMIG_N"])
        return out

    def live_columns(self, schema: str, tables: list[str], *,
                     chunk: int = 200) -> dict[str, dict[str, str]]:
        """`{table: {column: type}}` for many tables, from INFORMATION_SCHEMA.

        The copy compares the LIVE source's columns with the target's before
        it moves a row. The connector's table read answered that from its
        own metadata lookup, at 126-241 s per table (2026-09-29); one
        INFORMATION_SCHEMA.COLUMNS query answers a chunk of tables instead.
        Types are what the copy's pre-flight compares: NUMBER(p,s) becomes
        `decimal(p,s)` (the DECIMAL check), everything else is Snowflake's
        own type name lower-cased. A table the query does not list is ABSENT
        from the result, never an empty list -- the caller says so.
        Connector mode only.
        """
        out: dict[str, dict[str, str]] = {}
        database = self.database()
        if not database:
            raise SourceConfigError(
                "reading INFORMATION_SCHEMA needs the source database; add "
                "`database:` to the source config")
        for start in range(0, len(tables), chunk):
            batch = tables[start:start + chunk]
            names = ", ".join(f"'{_sql_literal(t)}'" for t in batch)
            sql = ("select TABLE_NAME, COLUMN_NAME, DATA_TYPE, "
                   "NUMERIC_PRECISION, NUMERIC_SCALE, ORDINAL_POSITION "
                   f"from {_sql_database(database)}.INFORMATION_SCHEMA.COLUMNS "
                   f"where TABLE_SCHEMA = '{_sql_literal(schema)}' "
                   f"and TABLE_NAME in ({names}) "
                   "order by TABLE_NAME, ORDINAL_POSITION")
            rows = sorted((r.asDict() for r in
                           self.pushdown(sql, schema=schema).collect()),
                          key=lambda d: (str(d["TABLE_NAME"]),
                                         int(d["ORDINAL_POSITION"] or 0)))
            for data in rows:
                kind = str(data.get("DATA_TYPE") or "").strip()
                if kind.upper() in ("NUMBER", "DECIMAL", "NUMERIC") and \
                        data.get("NUMERIC_PRECISION") is not None:
                    kind = (f"decimal({int(data['NUMERIC_PRECISION'])},"
                            f"{int(data.get('NUMERIC_SCALE') or 0)})")
                out.setdefault(str(data["TABLE_NAME"]), {})[
                    str(data["COLUMN_NAME"])] = kind.lower()
        return out

    def read_columns(self, schema: str, table: str,
                     select: list[tuple[str, str]]):
        """ONE qualified pushdown: `SELECT <expr> AS "<name>", ... FROM
        "DB"."SCHEMA"."TABLE"`.

        `select` is `[(name, read_expr)]`: a Snowflake expression over the
        quoted source column, unaliased. Each is aliased to the column's
        exact name, so the Spark side sees the source's names. Live
        2026-09-29 this cost ~8.5 s a table where the connector's table read
        cost 126-241 s, and the exact reads it carries (`::VARCHAR`,
        `TO_VARCHAR(.., FF9)`, `::ARRAY::VARCHAR`) are what keep NUMBER,
        TIME/TIMESTAMP fractions and VECTOR/MAP/OBJECT columns intact.
        The whole statement goes through the read-only guard, as every
        pushdown does.
        """
        if not select:
            raise SourceConfigError(f"no columns to read from {schema}.{table}")
        items = ", ".join(f"{expr} as {_sql_ident(name)}"
                          for name, expr in select)
        return self.pushdown(
            f"select {items} from {self.qualified(schema, table)}",
            schema=schema)

    def source_sums(self, schema: str, table: str,
                    columns: list[str]) -> dict[str, str | None]:
        """`{column: exact total as text}` summed IN SNOWFLAKE, one pushdown.

        `select sum("C")::VARCHAR as "C", ... from "DB"."SCHEMA"."TABLE"`:
        Snowflake adds its own NUMBERs exactly, and `::VARCHAR` carries the
        total past the connector's typing, which cut NUMBER to ten
        significant digits live (2026-09-29). None is SQL NULL -- no rows,
        or only NULLs.
        """
        if not columns:
            return {}
        items = ", ".join(f"sum({_sql_ident(c)})::VARCHAR as {_sql_ident(c)}"
                          for c in columns)
        row = self.pushdown(
            f"select {items} from {self.qualified(schema, table)}",
            schema=schema).collect()[0].asDict()
        return {c: (None if row.get(c) is None else str(row[c]))
                for c in columns}

    def register_columns_view(self, schema: str, table: str, view: str,
                              select: list[tuple[str, str]]) -> str:
        """`read_columns` registered as a session temp view; its name."""
        self.read_columns(schema, table, select).createOrReplaceTempView(view)
        return q(view)

    def read_table(self, schema: str, table: str):
        """A DataFrame over one source table."""
        if self.mode == "external-catalog":
            return self.spark.table(
                f"{q(self.external_catalog)}.{q(schema)}.{q(table)}")
        return (self.spark.read.format(AIDP_FORMAT)
                .options(**self._options)
                .option("schema", schema)
                .option("table", table)
                .load())

    def register_temp_view(self, schema: str, table: str, view: str) -> str:
        """Expose a source table to SQL as a temp view, and return its name.

        INSERT ... SELECT needs the source addressable in SQL. In
        external-catalog mode the three-part name already is; in connector
        mode the DataFrame is registered as a session-local temp view, which
        is dropped by the caller.
        """
        if self.mode == "external-catalog":
            return f"{q(self.external_catalog)}.{q(schema)}.{q(table)}"
        self.read_table(schema, table).createOrReplaceTempView(view)
        return q(view)

    def drop_temp_view(self, view: str) -> None:
        if self.mode == "connector":
            self.spark.catalog.dropTempView(view)

    def describe(self) -> dict:
        """What this source is, for the report header. Never the credential."""
        out = {"mode": self.mode}
        if self.mode == "external-catalog":
            out["external_catalog"] = self.external_catalog
        else:
            out.update(session_schema=self.session_schema,
                       host=self._options.get("host"),
                       database=self._options.get("database.name"),
                       user=self._options.get("user.name"),
                       warehouse=self._options.get("warehouse"),
                       role=self._options.get("role"),
                       auth=self._options.get("authentication.method"))
        return out


# What `provision` puts under a plan file's plain name when the file itself
# was too large to upload and went up gzipped beside it (see
# target/provisioning.py COMPRESS_OVER_BYTES).
PLAN_POINTER_KEY = "snowmig_compressed_to"


def read_plan_json(path) -> dict:
    """A plan file `provision` pushed: plain JSON, or a pointer to its gzip
    copy in the same folder, which is read and checked against the digest.

    Raises ValueError naming the file when what is there is not a plan.
    Live 2026-09-29, a failed overwrite upload left ddl_plan.json EMPTY on
    the workspace, and the stage died on a bare JSONDecodeError at char 0.
    """
    path = pathlib.Path(path)
    raw = path.read_bytes()
    try:
        doc = json.loads(raw.decode("utf-8"))
    except ValueError as exc:
        raise ValueError(
            f"{path} is not valid JSON ({len(raw):,} bytes: {exc}). A failed "
            f"upload leaves the file empty or partial; re-run `provision "
            f"--execute` and check that its upload of this file succeeded"
        ) from None
    if not (isinstance(doc, dict) and doc.get(PLAN_POINTER_KEY)):
        return doc
    name = str(doc[PLAN_POINTER_KEY])
    if pathlib.PurePosixPath(name).name != name or "\\" in name:
        raise ValueError(f"{path}: the pointer names {name!r}, which is not "
                         f"a file beside it")
    target = path.with_name(name)
    try:
        data = gzip.decompress(target.read_bytes())
    except (OSError, EOFError) as exc:
        raise ValueError(f"{target} (the plan {path.name} points at) could "
                         f"not be read: {exc}; re-run `provision --execute`"
                         ) from None
    if doc.get("sha256") and hashlib.sha256(data).hexdigest() != doc["sha256"]:
        raise ValueError(f"{target} does not match the digest its pointer "
                         f"{path.name} records: the two were not pushed "
                         f"together; re-run `provision --execute`")
    return json.loads(data.decode("utf-8"))


def write_step_output(output_dir: str | None, name: str,
                      payload: dict) -> str | None:
    """Save one runbook step's values under the workspace's report/output.

    `name` carries the step (S06_discover.json, S10_structure.json, ...).
    An empty output_dir switches it off. Never raises: the step's own
    reports are already written, and a copy that cannot be made must not
    turn a successful stage into a failed one.
    """
    if not output_dir:
        return None
    import datetime as _dt
    import json as _json
    try:
        folder = pathlib.Path(output_dir)
        folder.mkdir(parents=True, exist_ok=True)
        path = folder / name
        path.write_text(_json.dumps(
            {**payload, "written_at": _dt.datetime.now(
                _dt.timezone.utc).isoformat()}, indent=2, default=str))
        print(f"[output] {name} -> {path}", flush=True)
        return str(path)
    except OSError as exc:
        print(f"[output] {name} NOT written to {output_dir}: {exc}", flush=True)
        return None


## Stage logic

In [ ]:
#!/usr/bin/env python3
"""Plan vs reality: what landed in the target catalog, what did not, and why.

Runs on AIDP compute; READS ONLY. Joins three sources of truth —

  * `discovery_manifest.json`  what the external catalog exposed,
  * `structure_report_*.json` / `copy_report_*.json`  what the scripts claim,
  * the TARGET CATALOG itself  what actually exists (SHOW TABLES, and counts
    with --counts, one batched UNION ALL of three-part names per 50 tables) —

into `reconciliation.json` and `MIGRATION_REPORT.md`: one row per table with
its structure status, copy status, live existence, and live row count. The
catalog is consulted directly so the report cannot be flattered by a stale
script report: an object a report calls verified but the catalog no longer
holds -- or, with --counts, no longer holds at the verified row count -- is
flagged, and an object in the catalog that no report claims is flagged the
other way. A script report written for a DIFFERENT target -- another
catalog, or another schema than the one the approved plan names -- is
ignored (and said so), not applied to this one.

The target schema is resolved as 01_create_structure and 02_copy_schema
resolve it: the approved plan's `target_fqn` for the source schema, and only
where the plan is silent the one the reports recorded (else the source
schema's own name). Schema names compare case-insensitively, as Spark
resolves them.

"Could not look" never renders as zero: an unreadable schema is marked
UNREADABLE, distinct from empty.

Views are listed per schema, never omitted, with what the structure report
records for them: 01_create_structure --mode ddl-plan creates the plan's
views, so a view it created is VIEW_CREATED and one whose CREATE failed is
VIEW_FAILED (a problem verdict); a manifest view the plan does not carry is
VIEW_NOT_IN_PLAN, a planned one not created yet (dry run) is
VIEW_NOT_CREATED_YET, and one no ddl-plan run recorded (--mode ctas or
manifest, or no structure report) is VIEW_NOT_CREATED_BY_THIS_PATH, with the
report saying how views do get created.

A dynamic table or materialized view the approved plan migrates as a TABLE
SNAPSHOT (a TABLE statement with `snapshot_of`) is reconciled as a table --
structure, copy, existence, count -- wherever the manifest lists it:
discovery files a materialized view under `views`, and live 2026-09-29 this
report listed the snapshot as a view nobody created.
"""
from __future__ import annotations

import argparse
import datetime
import json
import pathlib
import sys

# /Workspace is the live-verified mount of the workspace tree on cluster
# filesystems (probed 2026-09-16 on a real cluster).

DEFAULT_REPORTS_DIR = "/Workspace/backup-snowflake-migration/reports"
# The step's own values also go here, next to the accumulated run report
# the CLI publishes after every stage. `--output-dir ''` switches it off.
DEFAULT_OUTPUT_DIR = "/Workspace/report/output"
MANIFEST_NAME = "discovery_manifest.json"

# Verdicts that mean something is WRONG, as opposed to not done yet. A
# migration runs schema by schema, so "this table was never attempted" is the
# normal state of most of the estate for most of the project -- exiting
# non-zero on it would make every partial run look broken, which is how a
# real signal gets ignored.
PROBLEM_VERDICTS = ("MISSING_DESPITE_REPORT", "VIEW_MISSING_DESPITE_REPORT",
                    "STRUCTURE_FAILED",
                    "STRUCTURE_TYPE_DRIFT", "STRUCTURE_ONLY_COPY_FAILED",
                    "COUNT_DRIFT", "TARGET_UNREADABLE", "VIEW_FAILED")

# A view's structure status -> its verdict. Anything else (no record, or
# `not_created_by_this_path` from --mode ctas / manifest) is
# VIEW_NOT_CREATED_BY_THIS_PATH.
_VIEW_VERDICTS = {"created": "VIEW_CREATED", "failed": "VIEW_FAILED",
                  "not_in_plan": "VIEW_NOT_IN_PLAN",
                  "dry_run": "VIEW_NOT_CREATED_YET",
                  "not_attempted": "VIEW_NOT_CREATED_YET"}


def q(identifier: str) -> str:
    return "`" + str(identifier).replace("`", "``") + "`"


def log(msg: str) -> None:
    print(f"[reconcile] {msg}", flush=True)


def fail(msg: str) -> int:
    """Report a refusal on BOTH streams and return 1.

    A notebook task captures stdout only: live, a script that exited 1 via a
    stderr-only message produced a job failure with NO explanation anywhere.
    """
    print(f"ERROR: {msg}", flush=True)
    print(f"error: {msg}", file=sys.stderr)
    return 1


def _load(reports: pathlib.Path, name: str) -> dict | None:
    path = reports / name
    return read_report_json(path) if path.exists() else None


def _for_catalog(report: dict | None,
                 target_catalog: str) -> tuple[dict | None, str | None]:
    """`report` if it was written for `target_catalog`, else (None, target).

    A report is evidence about the catalog it was written against and no
    other: 01 and 02 already refuse to resume from a report whose `target`
    differs, and the reports directory is shared, so after re-pointing at
    another catalog a copy report verified against the old one is still on
    disk. A report with no `target` (older shape) stays trusted, as 02 does.
    Catalog names compare case-insensitively, as Spark resolves them.
    """
    if not report:
        return None, None
    target = str(report.get("target") or "")
    if target and target.split(".", 1)[0].lower() != target_catalog.lower():
        return None, target
    return report, None


def _for_schema(report: dict | None,
                target_schema: str) -> tuple[dict | None, str | None]:
    """`report` if it was written for `target_schema`, else (None, target).

    The catalog check alone let a copy report verified against `lake.CORE`
    (the pre-fix copy's schema) certify `lake.db_core`, where the plan put
    the tables and where they held 0 rows: MIGRATED_VERIFIED, exit 0.
    """
    if not report:
        return report, None
    target = str(report.get("target") or "")
    if "." in target and \
            target.split(".", 1)[1].casefold() != target_schema.casefold():
        return None, target
    return report, None


def plan_targets(ddl_plan: dict,
                 target_catalog: str) -> dict[str, set[str]]:
    """`{source_schema: {target_schema}}` from the plan's TABLE statements
    in `target_catalog` -- the same `target_fqn` 01 creates the tables at."""
    out: dict[str, set[str]] = {}
    for stmt in ddl_plan.get("statements") or []:
        source = str(stmt.get("source_identifier") or "").split(".")
        target = str(stmt.get("target_fqn") or "").split(".")
        if len(source) != 3 or len(target) != 3:
            continue
        if str(stmt.get("object_type") or "TABLE").upper() == "VIEW":
            continue
        if target[0].casefold() != target_catalog.casefold():
            continue
        out.setdefault(source[1], set()).add(target[1])
    return out


def plan_views(ddl_plan: dict, target_catalog: str) -> dict[str, set[str]]:
    """`{source_schema: {view}}` for the plan's VIEW statements in
    `target_catalog`: the views 01_create_structure --mode ddl-plan creates.
    A table snapshot is a TABLE statement, so it is not one of them."""
    out: dict[str, set[str]] = {}
    for stmt in ddl_plan.get("statements") or []:
        source = str(stmt.get("source_identifier") or "").split(".")
        target = str(stmt.get("target_fqn") or "").split(".")
        if len(source) != 3 or len(target) != 3:
            continue
        if str(stmt.get("object_type") or "").upper() != "VIEW":
            continue
        if target[0].casefold() != target_catalog.casefold():
            continue
        out.setdefault(source[1], set()).add(source[2])
    return out


# A table in the target that no structure report records. Live 2026-09-29
# (50k-table estate): a structure run stopped after an hour had created 16
# tables after its last report write, and they read as verified structure.
UNRECORDED_REASON = (
    "in the target, but no structure report records it: a structure run "
    "stopped after creating it and before writing its report, or it was "
    "created outside this migration. Its layout has NOT been checked "
    "against the plan; re-run 01_create_structure, which checks it and "
    "records it")
PLANNED_VIEW_REASON = (
    "in the approved plan; no structure run has recorded it yet. "
    "01_create_structure --mode ddl-plan creates the plan's views after "
    "every table")


def plan_snapshots(ddl_plan: dict,
                   target_catalog: str) -> dict[str, list[str]]:
    """`{source_schema: [name]}` the plan migrates as a TABLE SNAPSHOT in
    `target_catalog`: tables here, whatever list the manifest put them in."""
    out: dict[str, list[str]] = {}
    for stmt in ddl_plan.get("statements") or []:
        source = str(stmt.get("source_identifier") or "").split(".")
        target = str(stmt.get("target_fqn") or "").split(".")
        if len(source) != 3 or len(target) != 3 or not stmt.get("snapshot_of"):
            continue
        if str(stmt.get("object_type") or "TABLE").upper() == "VIEW":
            continue
        if target[0].casefold() != target_catalog.casefold():
            continue
        names = out.setdefault(source[1], [])
        if source[2] not in names:
            names.append(source[2])
    return out


def _target_schema(schema: str, planned: set[str],
                   reports: list[dict]) -> str:
    """The plan's schema; where the plan is silent, the recorded one."""
    recorded = [str(r.get("target")).split(".", 1)[1] for r in reports
                if r and "." in str(r.get("target") or "")]
    if planned:
        for name in sorted(planned):
            if any(name.casefold() == r.casefold() for r in recorded):
                return name
        return sorted(planned)[0]
    return recorded[0] if recorded else schema


# What Spark says when a table, or the schema holding it, is simply not
# there -- the same markers 02_copy_schema reads as "absent" (a test pins the
# two lists equal). Only these mean "absent"; any other error is "could not
# look".
_NOT_FOUND = ("TABLE_OR_VIEW_NOT_FOUND", "SCHEMA_NOT_FOUND",
              "NoSuchTableException", "NoSuchNamespaceException",
              "NoSuchDatabaseException", "Table or view not found")


def _live_tables(spark, catalog: str, schema: str) -> set[str] | None:
    """Lower-cased table names the catalog holds; an EMPTY set when the
    schema does not exist yet; None when it could not be read.

    A schema the target has not created is a schema not migrated yet. Every
    SHOW TABLES error used to read as "could not look", so live, a
    schema-by-schema run exited 1 with every not-yet-created schema
    TARGET_UNREADABLE -- after each schema but the last.
    """
    try:
        rows = spark.sql(f"SHOW TABLES IN {q(catalog)}.{q(schema)}").collect()
    except Exception as exc:
        text = str(exc).lower()
        if any(marker.lower() in text for marker in _NOT_FOUND):
            return set()
        return None
    out = set()
    for r in rows:
        d = {k.lower(): v for k, v in r.asDict().items()}
        out.add(str(d.get("tablename") or d.get("name") or "").lower())
    return out


def _live_views(spark, catalog: str, schema: str) -> set[str] | None:
    """Lower-cased view names from SHOW VIEWS; None when it could not be
    read (not every catalog supports it)."""
    try:
        rows = spark.sql(f"SHOW VIEWS IN {q(catalog)}.{q(schema)}").collect()
    except Exception:
        return None
    out = set()
    for r in rows:
        d = {k.lower(): v for k, v in r.asDict().items()}
        out.add(str(d.get("viewname") or d.get("tablename")
                    or d.get("name") or "").lower())
    return out


def _view_exists(spark, fqn: str, name: str, live: set[str] | None,
                 views: set[str] | None) -> bool | None:
    """Whether the view is in the target -- LOOKED FOR, not taken from the
    structure report. True/False, or None when it could not be established.

    SHOW TABLES lists views on some catalogs; SHOW VIEWS on others; failing
    both, the view itself is described. A not-found answer is "no"; any
    other error is "could not look".
    """
    if live is not None and name in live:
        return True
    if views is not None:
        return name in views
    try:
        spark.sql(f"DESCRIBE TABLE {fqn}").collect()
        return True
    except Exception as exc:
        text = str(exc).lower()
        if any(marker.lower() in text for marker in _NOT_FOUND):
            return False
        return None


# Tables per batched count: one UNION ALL of fully qualified COUNT(*)s.
# 50 is the size the copy's batched source count was live-verified at
# (25 s, 2026-09-29).
COUNT_CHUNK = 50


def _target_counts(spark, catalog: str, schema: str,
                   tables: list[str]) -> dict[str, tuple[int | None, str | None]]:
    """`{table: (count, None) | (None, error)}`, ONE query per chunk.

    A COUNT(*) per table was a Spark job per table. Each branch is a
    three-part name tagged with its POSITION, so no table name has to
    survive being a string literal. One unreadable table fails the whole
    query and a batch cannot say which, so that chunk falls back to a count
    per table: the unreadable one keeps its own error, the rest are counted.
    """
    out: dict[str, tuple[int | None, str | None]] = {}

    def fqn(table: str) -> str:
        return f"{q(catalog)}.{q(schema)}.{q(table)}"

    for start in range(0, len(tables), COUNT_CHUNK):
        chunk = tables[start:start + COUNT_CHUNK]
        sql = " UNION ALL ".join(
            f"SELECT {i} AS `i`, COUNT(*) AS `n` FROM {fqn(t)}"
            for i, t in enumerate(chunk))
        try:
            got = {int(r["i"]): int(r["n"]) for r in spark.sql(sql).collect()}
            if sorted(got) != list(range(len(chunk))):
                raise RuntimeError(f"the batched count answered {len(got)} "
                                   f"of {len(chunk)} table(s)")
            for i, table in enumerate(chunk):
                out[table] = (got[i], None)
            continue
        except Exception as exc:
            log(f"{schema}: batched count of {len(chunk)} table(s) failed "
                f"({str(exc)[:120]}); counting them one by one")
        for table in chunk:
            try:
                out[table] = (spark.sql(
                    f"SELECT COUNT(*) AS n FROM {fqn(table)}"
                ).collect()[0]["n"], None)
            except Exception as exc:
                out[table] = (None, str(exc)[:200])
    return out


def reconcile(spark, *, manifest: dict, target_catalog: str,
              reports: pathlib.Path, counts: bool,
              planned_targets: dict[str, set[str]] | None = None,
              planned_snapshots: dict[str, list[str]] | None = None,
              planned_views: dict[str, set[str]] | None = None) -> dict:
    out = {"target_catalog": target_catalog,
           "generated_at": datetime.datetime.now(
               datetime.timezone.utc).isoformat(),
           "schemas": [], "totals": {}}
    tally: dict[str, int] = {}

    for schema_rec in manifest["schemas"]:
        schema = schema_rec["name"]
        # The plan's table snapshots are tables: added to the manifest's
        # tables, and taken out of its views (where a materialized view is).
        snapshots = list((planned_snapshots or {}).get(schema) or [])
        listed = {t["name"] for t in schema_rec["tables"]}
        tables = list(schema_rec["tables"]) + [
            {"name": n} for n in snapshots if n not in listed]
        manifest_views = [v for v in schema_rec.get("views") or []
                          if v["name"] not in snapshots]
        structure, s_other = _for_catalog(
            _load(reports, f"structure_report_{schema.lower()}.json"),
            target_catalog)
        copy, c_other = _for_catalog(
            _load(reports, f"copy_report_{schema.lower()}.json"),
            target_catalog)
        # The structure report first: it records where 01 created the
        # tables, which is where the copy has to have put the rows.
        target_schema = _target_schema(
            schema, (planned_targets or {}).get(schema) or set(),
            [structure, copy])
        structure, s_wrong = _for_schema(structure, target_schema)
        copy, c_wrong = _for_schema(copy, target_schema)
        ignored = {kind: other for kind, other in
                   (("structure", s_other or s_wrong),
                    ("copy", c_other or c_wrong)) if other}
        for kind, other in ignored.items():
            log(f"{schema}: the {kind} report targets {other}, not "
                f"{target_catalog}.{target_schema} — ignored for this target")
        live = _live_tables(spark, target_catalog, target_schema)
        live_counts = (_target_counts(
            spark, target_catalog, target_schema,
            [t["name"] for t in tables
             if t["name"].lower() in live])
            if counts and live is not None else {})

        rows = []
        for table in tables:
            name = table["name"]
            s_status = ((structure or {}).get("objects", {})
                        .get(name, {}).get("status", "not_attempted"))
            c_rec = (copy or {}).get("tables", {}).get(name, {})
            c_status = c_rec.get("status", "not_attempted")
            exists = (None if live is None else name.lower() in live)
            reason = None

            if live is None:
                verdict = "TARGET_UNREADABLE"
            elif not exists and (s_status in ("created", "already_existed")
                                 or c_status in ("verified",
                                                 "verified_with_conversion")):
                verdict = "MISSING_DESPITE_REPORT"
            elif s_status == "failed":
                # The CREATE raised. Whether or not something by that name
                # is there now, nobody has checked it: the operator has to
                # act, so this is not "never attempted". `not_in_plan` and
                # `dry_run` are intentional, and not problems.
                verdict = "STRUCTURE_FAILED"
            elif not exists and s_status == "not_in_plan":
                # Left out of the approved plan (an S9 scope reduction, or a
                # table the engine blocked): not pending. Live (2026-09-29)
                # a 4-table plan reported its 996 out-of-scope tables as
                # "not migrated yet -- expected while the migration is still
                # running", which they never will be under that plan.
                verdict = "NOT_IN_PLAN"
            elif not exists:
                verdict = "NOT_MIGRATED"
            elif s_status == "type_drift":
                # The table is there with a layout the plan did not produce.
                # A copy into it can verify counts and still have landed rows
                # in the wrong columns, so this outranks any copy status.
                verdict = "STRUCTURE_TYPE_DRIFT"
            elif c_status == "target_missing":
                # The copy found no table here, and the catalog lists one
                # now: the copy never ran against it. Falling through to
                # STRUCTURE_ONLY put "does not exist" beside "In target:
                # yes" under "No table is in a problem state".
                verdict = "STRUCTURE_ONLY_COPY_FAILED"
                reason = ("the copy recorded target_missing, but the "
                          "catalog lists it now: nothing was copied into "
                          "it. Re-run 02_copy_schema. (copy: "
                          + str(c_rec.get("reason") or "no reason") + ")")
            elif c_status == "verified":
                verdict = "MIGRATED_VERIFIED"
            elif c_status == "verified_with_conversion":
                # Counts verified, but a column was copied under a type the
                # plan never reviewed (mapping.source_type_drift: convert).
                # Opted into, so not a problem; never a plain pass either.
                verdict = "MIGRATED_WITH_CONVERSION"
                reason = "converted after a source type change: " + ", ".join(
                    f'{c} ({d.get("planned")} -> {d.get("live")}, into '
                    f'{d.get("target")})' for c, d in
                    (c_rec.get("source_type_drift") or {}).items())
            elif c_status in ("count_mismatch", "sum_mismatch",
                              "sum_not_comparable", "type_drift", "failed"):
                verdict = "STRUCTURE_ONLY_COPY_FAILED"
            elif c_status == "skipped_nonempty" and \
                    c_rec.get("source_count") is not None and \
                    c_rec.get("target_count") != c_rec.get("source_count"):
                # The record carries both counts and they disagree: the
                # status alone cannot make that a pass.
                verdict = "STRUCTURE_ONLY_COPY_FAILED"
            elif c_status == "skipped_nonempty":
                verdict = "PRESENT_NOT_REVERIFIED"
            else:
                verdict = "STRUCTURE_ONLY"
            # Only against a structure report for THIS target: with none
            # (tables made by `deploy`, which writes no report, or a report
            # for another target), "a structure run stopped after creating
            # it" would send the operator to the wrong fix.
            if (exists and verdict == "STRUCTURE_ONLY" and structure
                    and name not in structure.get("objects", {})):
                s_status, reason = "unrecorded", UNRECORDED_REASON

            row = {"table": name, "structure": s_status, "copy": c_status,
                   "exists_in_target": exists, "verdict": verdict,
                   "reason": reason or c_rec.get("reason")
                             or (structure or {}).get("objects", {})
                             .get(name, {}).get("reason")}
            if counts and exists:
                row["target_count"], error = live_counts.get(
                    name, (None, "not counted"))
                if error is not None:
                    row["count_error"] = error
                # The live count is compared, not just printed: a verified
                # table emptied or changed out of band since the copy is a
                # problem, not a pass. A count that could not be read is
                # not drift, and a report that never recorded one has
                # nothing to compare against.
                reported = c_rec.get("target_count")
                if verdict in ("MIGRATED_VERIFIED",
                               "MIGRATED_WITH_CONVERSION") \
                        and row["target_count"] is not None \
                        and isinstance(reported, int) \
                        and row["target_count"] != reported:
                    verdict = row["verdict"] = "COUNT_DRIFT"
                    row["reason"] = (f"the copy report verified {reported:,} "
                                     f"row(s); the target now holds "
                                     f"{row['target_count']:,}. Changed "
                                     f"since the copy, not by it")
            rows.append(row)
            tally[verdict] = tally.get(verdict, 0) + 1

        # Views: every manifest view is listed rather than silently absent,
        # with what 01 recorded for it. This used to read a hard-coded
        # VIEW_NOT_CREATED_BY_THIS_PATH, so a view 01 had FAILED to create
        # came out under "No table is in a problem state", exit 0. SHOW
        # TABLES lists views on some catalogs and not others, and nothing
        # here looks for views specifically, so "not listed" is None (could
        # not look), never "no".
        view_rows = []
        live_views = None          # read once per schema, only if needed
        s_views = (structure or {}).get("views") or {}
        s_objects = (structure or {}).get("objects") or {}
        for view in manifest_views:
            name = view["name"]
            s_rec = s_views.get(name)
            if s_rec is None and str((s_objects.get(name) or {})
                                     .get("kind") or "").upper() == "VIEW":
                # A report written while views were recorded in `objects`.
                s_rec = s_objects[name]
            s_rec = s_rec or {}
            s_view = s_rec.get("status", "not_attempted")
            v_reason = s_rec.get("reason")
            if live is None:
                v_exists, v_verdict = None, "TARGET_UNREADABLE"
            elif not s_rec and name in (planned_views or {}).get(schema, ()):
                # Planned, and no run has recorded it: a ddl-plan run that
                # has not reached its view phase, not a path that never
                # creates views.
                v_exists = True if name.lower() in live else None
                v_verdict, v_reason = "VIEW_NOT_CREATED_YET", PLANNED_VIEW_REASON
            else:
                v_exists = True if name.lower() in live else None
                v_verdict = ((_VIEW_VERDICTS.get(s_view)
                              if s_rec or s_view != "not_attempted" else None)
                             or "VIEW_NOT_CREATED_BY_THIS_PATH")
            if v_verdict == "VIEW_CREATED":
                # The structure report says created; the target has to
                # agree. A view dropped since, or created somewhere else,
                # used to read VIEW_CREATED and exit 0 on 01's word alone.
                t_fqn = str(s_rec.get("target_fqn") or "")
                written = t_fqn.rsplit(".", 1)[-1] if t_fqn else name
                t_name = written.lower()     # listings are compared folded
                if live_views is None and not (t_name in (live or set())):
                    live_views = _live_views(spark, target_catalog,
                                             target_schema)
                v_exists = _view_exists(
                    spark, f"{q(target_catalog)}.{q(target_schema)}."
                           f"{q(written)}", t_name, live, live_views)
                if v_exists is False:
                    v_verdict = "VIEW_MISSING_DESPITE_REPORT"
                    v_reason = ("the structure report records it created, "
                                "but the target does not have it: dropped "
                                "since, or created elsewhere. Re-run "
                                "01_create_structure")
                elif v_exists is None:
                    v_verdict = "TARGET_UNREADABLE"
                    v_reason = ("the structure report records it created, "
                                "but whether the target has it could not be "
                                "read (SHOW TABLES, SHOW VIEWS and DESCRIBE "
                                "all failed)")
            view_rows.append({"view": name, "structure": s_view,
                              "exists_in_target": v_exists,
                              "verdict": v_verdict,
                              "reason": v_reason})
            tally[v_verdict] = tally.get(v_verdict, 0) + 1

        known = ({t["name"].lower() for t in tables}
                 | {v["name"].lower() for v in manifest_views})
        unclaimed = sorted(live - known) if live is not None else []
        out["schemas"].append({
            "schema": schema, "target_schema": target_schema,
            "target_readable": live is not None,
            "tables": rows,
            "views": view_rows,
            "in_target_but_not_in_manifest": unclaimed,
            "reports_ignored_for_other_catalog": ignored})

    out["totals"] = tally
    return out


def render(rec: dict) -> str:
    totals = rec["totals"]
    problems = sum(totals.get(v, 0) for v in PROBLEM_VERDICTS)
    lines = [
        "# Migration report — plan vs what the target catalog actually holds",
        "",
        f'Target: `{rec["target_catalog"]}` · generated {rec["generated_at"]}',
        "",
        "Verdicts: " + ", ".join(f'{k} = {v}'
                                 for k, v in sorted(totals.items())),
        "",
        (f'**{problems} object(s) need attention** '
         f'({", ".join(PROBLEM_VERDICTS)}).' if problems else
         "No table is in a problem state. Anything below that is not "
         "migrated simply has not been attempted yet — a migration runs "
         "schema by schema, so that is the expected middle of the project, "
         "not a failure."),
        "",
    ]
    for s in rec["schemas"]:
        lines += [f'## `{s["schema"]}` → `{rec["target_catalog"]}.'
                  f'{s["target_schema"]}`', ""]
        if not s["target_readable"]:
            lines += ["**Target schema UNREADABLE — nothing below is "
                      "confirmed, and this is not the same as empty.**", ""]
        for kind, other in (s.get("reports_ignored_for_other_catalog")
                            or {}).items():
            lines += [f"The {kind} report on disk targets `{other}`, not "
                      f"this target — ignored here.", ""]
        lines += ["| Table | Structure | Copy | In target | Verdict | Why |",
                  "|---|---|---|---|---|---|"]
        for t in s["tables"]:
            exists = {True: "yes", False: "**no**", None: "?"}[t["exists_in_target"]]
            count = (f' ({t["target_count"]:,} rows)'
                     if t.get("target_count") is not None else "")
            reason = (t.get("reason") or "").replace("|", "\\|")[:120]
            lines.append(f'| {t["table"]}{count} | {t["structure"]} | '
                         f'{t["copy"]} | {exists} | {t["verdict"]} | {reason} |')
        if s.get("views"):
            how = ("01_create_structure --mode ddl-plan creates the views "
                   "the approved plan carries, after every table.")
            if any(v["verdict"] == "VIEW_NOT_CREATED_BY_THIS_PATH"
                   for v in s["views"]):
                how += (" No ddl-plan run recorded the views marked "
                        "VIEW_NOT_CREATED_BY_THIS_PATH (--mode ctas and "
                        "manifest create tables only); create them with "
                        "that stage or `snowmig deploy --execute` (catalog "
                        "API) and verify them against the source.")
            lines += ["", "### Views", "",
                      how + " \"Listed\" is what SHOW TABLES returned; `?` "
                      "means it was not looked for.", "",
                      "| View | Structure | Listed in target | Verdict | Why |",
                      "|---|---|---|---|---|"]
            for v in s["views"]:
                listed = {True: "yes", None: "?"}.get(v["exists_in_target"], "?")
                reason = (v.get("reason") or "").replace("|", "\\|")[:120]
                lines.append(f'| {v["view"]} | {v["structure"]} | {listed} | '
                             f'{v["verdict"]} | {reason} |')
        if s["in_target_but_not_in_manifest"]:
            lines += ["", f'⚠️ In the target but in no report: '
                          f'{", ".join(s["in_target_but_not_in_manifest"])} — '
                          f'someone else\'s objects, or a stale manifest.']
        lines.append("")
    return "\n".join(lines)


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--target-catalog", required=True)
    ap.add_argument("--output-dir", default=DEFAULT_OUTPUT_DIR,
                    help="where this step saves its values (report/output in "
                         "the workspace); '' to skip")
    ap.add_argument("--reports-dir", default=DEFAULT_REPORTS_DIR)
    ap.add_argument("--ddl-plan",
                    help="path to ddl_plan.json (default: ../plan/"
                         "ddl_plan.json next to --reports-dir); its "
                         "target_fqn names the target schema, as in "
                         "01_create_structure")
    ap.add_argument("--counts", action="store_true",
                    help="also read a live COUNT(*) per existing table (one "
                         "batched query per 50 tables), and "
                         "flag a verified table whose count has changed since "
                         "the copy verified it (COUNT_DRIFT)")
    args = ap.parse_args(argv)

    reports = pathlib.Path(args.reports_dir)
    manifest = _load(reports, MANIFEST_NAME)
    if manifest is None:
        return fail(f"error: {reports / MANIFEST_NAME} not found; run 00_discover "
              f"first")

    ddl_path = (pathlib.Path(args.ddl_plan) if args.ddl_plan
                else reports.parent / "plan" / "ddl_plan.json")
    planned = None
    snapshots = None
    views = None
    if ddl_path.is_file():
        try:
            ddl_plan = read_plan_json(ddl_path)
        except ValueError as exc:
            return fail(str(exc))
        planned = plan_targets(ddl_plan, args.target_catalog)
        snapshots = plan_snapshots(ddl_plan, args.target_catalog)
        views = plan_views(ddl_plan, args.target_catalog)
    elif args.ddl_plan:
        return fail(f"error: --ddl-plan {ddl_path} is not there")

    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

    rec = reconcile(spark, manifest=manifest,
                    target_catalog=args.target_catalog, reports=reports,
                    counts=args.counts, planned_targets=planned,
                    planned_snapshots=snapshots, planned_views=views)
    (reports / "reconciliation.json").write_text(json.dumps(rec, indent=2), encoding="utf-8")
    (reports / "MIGRATION_REPORT.md").write_text(render(rec), encoding="utf-8")
    log(f"totals: {rec['totals']}")
    log(f"-> {reports / 'MIGRATION_REPORT.md'}")
    write_step_output(args.output_dir, "S11_reconcile.json", {
        "step": "S11", "stage": "reconcile",
        "target_catalog": args.target_catalog, "totals": rec["totals"]})

    problems = sum(rec["totals"].get(v, 0) for v in PROBLEM_VERDICTS)
    pending = sum(v for k, v in rec["totals"].items()
                  if k not in PROBLEM_VERDICTS
                  and k not in ("MIGRATED_VERIFIED",
                                "MIGRATED_WITH_CONVERSION",
                                "PRESENT_NOT_REVERIFIED",
                                "VIEW_CREATED", "VIEW_NOT_IN_PLAN",
                                "VIEW_NOT_CREATED_BY_THIS_PATH",
                                "NOT_IN_PLAN"))
    scoped_out = rec["totals"].get("NOT_IN_PLAN", 0)
    if scoped_out:
        log(f"{scoped_out} table(s) are not in the approved plan "
            f"(NOT_IN_PLAN) -- left out by its scope, not pending.")
    if pending:
        log(f"{pending} table(s) not migrated yet — expected while the "
            f"migration is still running, schema by schema. Not an error.")
    if problems:
        log(f"{problems} object(s) in a PROBLEM state "
            f"({', '.join(PROBLEM_VERDICTS)}) — see the report.")
    return 1 if problems else 0



In [ ]:
# ── RUN ────────────────────────────────────────────────────────────
# main() RETURNS an exit code; it is not allowed to raise SystemExit here.
# A notebook cell that raises SystemExit is reported as a FAILED task even
# when the work succeeded, so the code is inspected and only a real failure
# is re-raised -- which keeps a genuinely failed stage failing.
code = main(ARGV)
print('exit code:', code, flush=True)
if code:
    raise RuntimeError(f'stage exited {code}')
